In [ ]:
!pip install albumentations timm einops


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
from torch.amp import autocast, GradScaler
import random
import copy

# EViT-UNet dependencies
import timm
from einops import rearrange

# Data augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4
print(f"Device: {DEVICE}")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

Device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE = "/content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation"

DATASETS = {
    "boreal": {
        "images": os.path.join(BASE, "boreal", "images"),
        "masks":  os.path.join(BASE, "boreal", "masks"),
    },
    "kaggle": {
        "images": os.path.join(BASE, "kaggle", "images"),
        "masks":  os.path.join(BASE, "kaggle", "masks"),
    },
    "corsican": {
        "images": os.path.join(BASE, "corsican", "images"),
        "masks":  os.path.join(BASE, "corsican", "masks"),
    }
}

print("Dataset paths:")
for name, paths in DATASETS.items():
    exists = os.path.isdir(paths["images"])
    print(f"  {name}: {'OK' if exists else 'NOT FOUND'} → {paths['images']}")

Dataset paths:
  boreal: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/boreal/images
  kaggle: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/kaggle/images
  corsican: OK → /content/drive/MyDrive/MachineLearningII/FireSemanticSegmentation/corsican/images


Dataset

In [ ]:
class FireDataset(Dataset):
    def __init__(self, images_path, masks_path, transform=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.transform = transform
        self.images = sorted(os.listdir(images_path))

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_path, img_name)

        stem = os.path.splitext(img_name)[0]
        mask_name = img_name.replace("_rgb", "_gt")
        mask_stem = os.path.splitext(mask_name)[0]

        for ext in [os.path.splitext(mask_name)[1], '.png', '.jpg', '.jpeg']:
            candidate = mask_stem + ext
            if os.path.exists(os.path.join(self.masks_path, candidate)):
                mask_name = candidate
                break

        mask_path = os.path.join(self.masks_path, mask_name)

        image = np.array(Image.open(img_path).convert("RGB"))
        mask  = np.array(Image.open(mask_path).convert("L"))
        mask  = (mask > 0).astype("float32")

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"].unsqueeze(0)

        if image.dim() == 3 and (image.shape[1] != image.shape[2] or image.shape[1] == 0):
            target = image.shape[1]
            image = nn.functional.interpolate(
                image.unsqueeze(0), size=(target, target), mode='bilinear', align_corners=False
            ).squeeze(0)

        return image, mask


Data Augmentation

In [ ]:
DATASET_IMG_SIZE = {
    'corsican': 256,
    'boreal':   256,
    'kaggle':   256,
}

def get_transforms(img_size):
    """Return (train_transform, val_transform) for a given resolution."""
    train = A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.HueSaturationValue(p=0.3),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    val = A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])
    return train, val

train_transform, val_transform = get_transforms(256)

Data Splits

In [ ]:
def create_data_splits(images_path, masks_path, train_transform, val_transform, data_percentage=1.0):
    """
    Create data splits with specific percentage of total dataset.

    Args:
        images_path: Path to images
        masks_path: Path to masks
        train_transform: Training transformations
        val_transform: Validation/test transformations
        data_percentage: Data split percentages (0.25, 0.5, 1.0)

    Returns:
        train_dataset, val_dataset, test_dataset
    """
    # Complete dataset
    total_size = len(FireDataset(images_path, masks_path))

    # Get size based on the percentage
    used_size = int(total_size * data_percentage)

    # Splits: 70% train, 15% val, 15% test
    train_size = int(0.7 * used_size)
    val_size = int(0.15 * used_size)
    test_size = used_size - train_size - val_size

    print(f"\nUsing {data_percentage*100:.0f}% of dataset:")
    print(f"  Total images: {total_size}")
    print(f"  Used images: {used_size}")
    print(f"  Train: {train_size}")
    print(f"  Val: {val_size}")
    print(f"  Test: {test_size}")

    # Create datasets
    full_train = FireDataset(images_path, masks_path, transform=train_transform)
    full_val = FireDataset(images_path, masks_path, transform=val_transform)
    full_test = FireDataset(images_path, masks_path, transform=val_transform)

    # Generar random indices
    indices = list(range(total_size))
    random.seed(42)
    random.shuffle(indices)

    # Take only the used percentage
    indices = indices[:used_size]

    # Divide in train/val/test
    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create subsets
    train_dataset = Subset(full_train, train_indices)
    val_dataset = Subset(full_val, val_indices)
    test_dataset = Subset(full_test, test_indices)

    return train_dataset, val_dataset, test_dataset

Dice Loss (Loss function)

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, preds, targets):
        preds = torch.sigmoid(preds)
        preds = preds.view(-1)
        targets = targets.view(-1)

        intersection = (preds * targets).sum()
        dice = (2. * intersection + self.smooth) / (
            preds.sum() + targets.sum() + self.smooth
        )

        return 1 - dice

Metrics

In [ ]:
def get_metrics(preds, targets, threshold=0.5):
    """Calcula F1, mIoU, MCC, HAF"""
    preds = torch.sigmoid(preds)
    preds = (preds > threshold).float()

    preds = preds.view(-1)
    targets = targets.view(-1)

    TP = (preds * targets).sum()
    TN = ((1 - preds) * (1 - targets)).sum()
    FP = (preds * (1 - targets)).sum()
    FN = ((1 - preds) * targets).sum()

    # F1 Score
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = (2 * precision * recall) / (precision + recall + 1e-8)

    # mIoU
    miou = TP / (TP + FP + FN + 1e-8)

    # MCC
    mcc = (TP * TN - FP * FN) / torch.sqrt(
        (TP + FP) * (TP + FN) * (TN + FP) * (TN + FN) + 1e-8
    )

    # HAF
    haf = TP / (TP + FP + FN + 1e-8)

    return f1.item(), miou.item(), mcc.item(), haf.item()

Model

In [ ]:
import timm


class EViTUNet(nn.Module):
    """EViT-UNet with PVT-v2-B2 pretrained encoder."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.encoder = timm.create_model('pvt_v2_b2', pretrained=pretrained, features_only=True)
        enc_chs = self.encoder.feature_info.channels()  # [64, 128, 320, 512]

        # 512+320 = 832, 256+128 = 384, 128+64 = 192, 64
        decoder_in  = [enc_chs[3] + enc_chs[2], 256 + enc_chs[1], 128 + enc_chs[0], 64]
        decoder_out = [256, 128, 64, 32]

        self.decoder_convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(decoder_in[i], decoder_out[i], kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(decoder_out[i]),
                nn.ReLU(inplace=True),
            )
            for i in range(4)
        ])
        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1, e2, e3, e4 = self.encoder(x)

        d = nn.functional.interpolate(e4, scale_factor=2, mode='bilinear', align_corners=False)
        d = torch.cat([d, e3], dim=1)
        d = self.decoder_convs[0](d)

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = torch.cat([d, e2], dim=1)
        d = self.decoder_convs[1](d)

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = torch.cat([d, e1], dim=1)
        d = self.decoder_convs[2](d)

        d = nn.functional.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
        d = self.decoder_convs[3](d)

        d = nn.functional.interpolate(d, size=(x.shape[2], x.shape[3]),
                                       mode='bilinear', align_corners=False)
        return self.head(d)


def build_evit_unet():
    return EViTUNet(pretrained=True)


_m = build_evit_unet()
_x = torch.zeros(2, 3, 256, 256)
_out = _m(_x)
print(f"EViT-UNet (PVT-v2-B2) output: {_out.shape}")
print(f"Encoder channels: {_m.encoder.feature_info.channels()}")
print(f"Total params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
print(f"Encoder params: {sum(p.numel() for p in _m.encoder.parameters())/1e6:.1f}M (pretrained)")
print(f"Decoder params: {sum(p.numel() for p in _m.decoder_convs.parameters())/1e6:.1f}M (random init)")
del _m, _x, _out


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/101M [00:00<?, ?B/s]

EViT-UNet (PVT-v2-B2) output: torch.Size([2, 1, 256, 256])
Encoder channels: [64, 128, 320, 512]
Total params:    27.3M
Encoder params:  24.8M (pretrained)
Decoder params:  2.5M (random init)


Epoch Training and Validation

In [ ]:
scaler = GradScaler('cuda')

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    totals = dict(loss=0, f1=0, miou=0, mcc=0, haf=0)

    for images, masks in loader:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        with autocast('cuda'):
            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, masks)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        with torch.no_grad():
            f1, miou, mcc, haf = get_metrics(logits, masks)
        totals['loss'] += loss.item()
        totals['f1'] += f1
        totals['miou'] += miou
        totals['mcc'] += mcc
        totals['haf'] += haf

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


def validate_epoch(model, loader, criterion):
    model.eval()
    totals = dict(loss=0, f1=0, miou=0, mcc=0, haf=0)

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = nn.functional.interpolate(logits, size=masks.shape[-2:], mode='bilinear', align_corners=False)
            loss = criterion(logits, masks)
            f1, miou, mcc, haf = get_metrics(logits, masks)
            totals['loss'] += loss.item()
            totals['f1'] += f1
            totals['miou'] += miou
            totals['mcc'] += mcc
            totals['haf'] += haf

    n = len(loader)
    return {k: v / n for k, v in totals.items()}


Experiment Configuration

In [ ]:
BATCH_SIZE = 8
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 15
LR = 1e-4
PERCENTAGES = [0.25, 0.5, 1.0]

# Store results
all_histories = {}
all_test_results = {}

print("Starting training with different dataset percentages...\n")

Starting training with different dataset percentages...



Training Model

In [ ]:
def train_model(train_loader, val_loader, test_loader, dataset_name, data_percentage, num_epochs=NUM_EPOCHS, early_stopping_patience=EARLY_STOPPING_PATIENCE):
    global scaler
    scaler = GradScaler('cuda')

    pct_label = f"{int(data_percentage*100)}%"
    ckpt_name = f"evitunet_{dataset_name}_{int(data_percentage*100)}pct.pth"

    print(f"\n{'='*70}")
    print(f"TRAINING WITH {data_percentage*100:.0f}% OF DATASET — {dataset_name.upper()}")
    print(f"{'='*70}\n")

    torch.manual_seed(SEED)
    model = build_evit_unet()
    model = model.to(DEVICE)

    criterion = DiceLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    history = {'train_loss': [], 'train_f1': [], 'train_miou': [], 'train_mcc': [], 'train_haf': [], 'val_loss': [], 'val_f1': [], 'val_miou': [], 'val_mcc': [], 'val_haf': []}

    best_val_miou = 0.0
    best_model_path = ckpt_name
    patience_counter = 0

    for epoch in range(num_epochs):
        tr = train_epoch(model, train_loader, criterion, optimizer)
        vl = validate_epoch(model, val_loader, criterion)

        history['train_loss'].append(tr['loss'])
        history['train_f1'].append(tr['f1'])
        history['train_miou'].append(tr['miou'])
        history['train_mcc'].append(tr['mcc'])
        history['train_haf'].append(tr['haf'])

        history['val_loss'].append(vl['loss'])
        history['val_f1'].append(vl['f1'])
        history['val_miou'].append(vl['miou'])
        history['val_mcc'].append(vl['mcc'])
        history['val_haf'].append(vl['haf'])

        # Print progress every 5 epochs
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}]")
            print(f"  Train - Loss: {tr['loss']:.4f}, F1: {tr['f1']:.4f}, mIoU: {tr['miou']:.4f}")
            print(f"  Val   - Loss: {vl['loss']:.4f}, F1: {vl['f1']:.4f}, mIoU: {vl['miou']:.4f}")

        if vl['miou'] > best_val_miou:
            best_val_miou = vl['miou']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_miou': vl['miou'],
                'metrics': {'f1': vl['f1'], 'miou': vl['miou'], 'mcc': vl['mcc'], 'haf': vl['haf']}
            }, best_model_path)
            if (epoch + 1) % 5 == 0:
                print(f"  ✓ Best model saved! (mIoU: {best_val_miou:.4f})")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= early_stopping_patience:
            print(f"  → Early stopping at epoch {epoch+1} "
                  f"(no improvement for {early_stopping_patience} epochs)")
            break

    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nBest model loaded from epoch {checkpoint['epoch']+1}")

    ts = validate_epoch(model, test_loader, criterion)

    print(f"\n{'='*70}")
    print(f"FINAL TEST RESULTS ({data_percentage*100:.0f}% dataset — {dataset_name.upper()})")
    print(f"{'='*70}")
    print(f"Loss:  {ts['loss']:.4f}")
    print(f"F1:    {ts['f1']:.4f}")
    print(f"mIoU:  {ts['miou']:.4f}")
    print(f"MCC:   {ts['mcc']:.4f}")
    print(f"HAF:   {ts['haf']:.4f}")
    print(f"{'='*70}\n")

    return history, {'test_loss': ts['loss'], 'test_f1': ts['f1'], 'test_miou': ts['miou'], 'test_mcc': ts['mcc'], 'test_haf': ts['haf'], 'epochs_run': epoch + 1}


Training — 3 datasets × 3 percentages

### Dataset: CORSICAN

In [ ]:
_img_size_corsican = DATASET_IMG_SIZE['corsican']
_tr_tf_corsican, _vl_tf_corsican = get_transforms(_img_size_corsican)
print(f"\nCORSICAN — resolution: {_img_size_corsican}x{_img_size_corsican}")

_, _val_ds_corsican, _test_ds_corsican = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=1.0
)
_val_loader_corsican = DataLoader(_val_ds_corsican,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_corsican = DataLoader(_test_ds_corsican, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



CORSICAN — resolution: 256x256

Using 100% of dataset:
  Total images: 500
  Used images: 500
  Train: 350
  Val: 75
  Test: 75


#### CORSICAN — 25% training data

In [ ]:
_train_ds_corsican_25, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=0.25
)
_train_loader_corsican_25 = DataLoader(
    _train_ds_corsican_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_25, _res_corsican_25 = train_model(
    _train_loader_corsican_25, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=0.25,
)

all_histories[('corsican', '25%')] = _hist_corsican_25
all_test_results[('corsican', '25%')] = _res_corsican_25



Using 25% of dataset:
  Total images: 500
  Used images: 125
  Train: 87
  Val: 18
  Test: 20

TRAINING WITH 25% OF DATASET — CORSICAN



Epoch [1/50]
  Train - Loss: 0.5887, F1: 0.6240, mIoU: 0.4734
  Val   - Loss: 0.5363, F1: 0.7635, mIoU: 0.6182
Epoch [5/50]
  Train - Loss: 0.4392, F1: 0.8588, mIoU: 0.7552
  Val   - Loss: 0.3718, F1: 0.8830, mIoU: 0.7907
Epoch [10/50]
  Train - Loss: 0.4116, F1: 0.8996, mIoU: 0.8183
  Val   - Loss: 0.3467, F1: 0.8912, mIoU: 0.8039
Epoch [15/50]
  Train - Loss: 0.4017, F1: 0.8987, mIoU: 0.8197
  Val   - Loss: 0.3299, F1: 0.9241, mIoU: 0.8590
Epoch [20/50]
  Train - Loss: 0.3856, F1: 0.9134, mIoU: 0.8426
  Val   - Loss: 0.3134, F1: 0.9134, mIoU: 0.8407
Epoch [25/50]
  Train - Loss: 0.3778, F1: 0.9124, mIoU: 0.8408
  Val   - Loss: 0.3027, F1: 0.9341, mIoU: 0.8764
Epoch [30/50]
  Train - Loss: 0.3500, F1: 0.9183, mIoU: 0.8502
  Val   - Loss: 0.2861, F1: 0.9301, mIoU: 0.8694
Epoch [35/50]
  Train - Loss: 0.3311, F1: 0.9299, mIoU: 0.8694
  Val   - Loss: 0.2731, F1: 0.9305, mIoU: 0.8702
Epoch [40/50]
  Train - Loss: 0.3155, F1: 0.9361, mIoU: 0.8803
  Val   - Loss: 0.2624, F1: 0.9324, mIoU: 0

#### CORSICAN — 50% training data

In [ ]:
_train_ds_corsican_50, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=0.5
)
_train_loader_corsican_50 = DataLoader(
    _train_ds_corsican_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_50, _res_corsican_50 = train_model(
    _train_loader_corsican_50, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=0.5,
)

all_histories[('corsican', '50%')] = _hist_corsican_50
all_test_results[('corsican', '50%')] = _res_corsican_50



Using 50% of dataset:
  Total images: 500
  Used images: 250
  Train: 175
  Val: 37
  Test: 38

TRAINING WITH 50% OF DATASET — CORSICAN

Epoch [1/50]
  Train - Loss: 0.5461, F1: 0.6915, mIoU: 0.5377
  Val   - Loss: 0.4155, F1: 0.7764, mIoU: 0.6349
Epoch [5/50]
  Train - Loss: 0.4270, F1: 0.8861, mIoU: 0.7970
  Val   - Loss: 0.3457, F1: 0.8992, mIoU: 0.8169
Epoch [10/50]
  Train - Loss: 0.4000, F1: 0.9018, mIoU: 0.8227
  Val   - Loss: 0.3134, F1: 0.9236, mIoU: 0.8582
Epoch [15/50]
  Train - Loss: 0.3647, F1: 0.9173, mIoU: 0.8479
  Val   - Loss: 0.2906, F1: 0.9197, mIoU: 0.8515
Epoch [20/50]
  Train - Loss: 0.3476, F1: 0.9147, mIoU: 0.8441
  Val   - Loss: 0.2700, F1: 0.9260, mIoU: 0.8623
Epoch [25/50]
  Train - Loss: 0.3098, F1: 0.9307, mIoU: 0.8709
  Val   - Loss: 0.2406, F1: 0.9342, mIoU: 0.8766
Epoch [30/50]
  Train - Loss: 0.2928, F1: 0.9346, mIoU: 0.8779
  Val   - Loss: 0.2247, F1: 0.9231, mIoU: 0.8575
Epoch [35/50]
  Train - Loss: 0.2570, F1: 0.9397, mIoU: 0.8869
  Val   - Loss: 0

#### CORSICAN — 100% training data

In [ ]:
_train_ds_corsican_100, _, _ = create_data_splits(
    DATASETS['corsican']['images'], DATASETS['corsican']['masks'],
    _tr_tf_corsican, _vl_tf_corsican, data_percentage=1.0
)
_train_loader_corsican_100 = DataLoader(
    _train_ds_corsican_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_corsican_100, _res_corsican_100 = train_model(
    _train_loader_corsican_100, _val_loader_corsican, _test_loader_corsican,
    dataset_name='corsican', data_percentage=1.0,
)

all_histories[('corsican', '100%')] = _hist_corsican_100
all_test_results[('corsican', '100%')] = _res_corsican_100



Using 100% of dataset:
  Total images: 500
  Used images: 500
  Train: 350
  Val: 75
  Test: 75

TRAINING WITH 100% OF DATASET — CORSICAN

Epoch [1/50]
  Train - Loss: 0.4915, F1: 0.7582, mIoU: 0.6245
  Val   - Loss: 0.3701, F1: 0.8683, mIoU: 0.7673
Epoch [5/50]
  Train - Loss: 0.3882, F1: 0.9011, mIoU: 0.8217
  Val   - Loss: 0.3209, F1: 0.9069, mIoU: 0.8298
Epoch [10/50]
  Train - Loss: 0.3314, F1: 0.9189, mIoU: 0.8509
  Val   - Loss: 0.2628, F1: 0.9317, mIoU: 0.8723
Epoch [15/50]
  Train - Loss: 0.2774, F1: 0.9276, mIoU: 0.8655
  Val   - Loss: 0.2247, F1: 0.9328, mIoU: 0.8741
Epoch [20/50]
  Train - Loss: 0.2305, F1: 0.9384, mIoU: 0.8845
  Val   - Loss: 0.1878, F1: 0.9390, mIoU: 0.8851
Epoch [25/50]
  Train - Loss: 0.1938, F1: 0.9434, mIoU: 0.8932
  Val   - Loss: 0.1558, F1: 0.9399, mIoU: 0.8867
Epoch [30/50]
  Train - Loss: 0.1592, F1: 0.9477, mIoU: 0.9009
  Val   - Loss: 0.1349, F1: 0.9438, mIoU: 0.8936
Epoch [35/50]
  Train - Loss: 0.1351, F1: 0.9525, mIoU: 0.9095
  Val   - Loss:

### Dataset: BOREAL

In [ ]:
_img_size_boreal = DATASET_IMG_SIZE['boreal']
_tr_tf_boreal, _vl_tf_boreal = get_transforms(_img_size_boreal)
print(f"\nBOREAL — resolution: {_img_size_boreal}x{_img_size_boreal}")

_, _val_ds_boreal, _test_ds_boreal = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=1.0
)
_val_loader_boreal = DataLoader(_val_ds_boreal,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_boreal = DataLoader(_test_ds_boreal, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



BOREAL — resolution: 256x256

Using 100% of dataset:
  Total images: 1417
  Used images: 1417
  Train: 991
  Val: 212
  Test: 214


#### BOREAL — 25% training data

In [ ]:
_train_ds_boreal_25, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=0.25
)
_train_loader_boreal_25 = DataLoader(
    _train_ds_boreal_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_25, _res_boreal_25 = train_model(
    _train_loader_boreal_25, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=0.25,
)

all_histories[('boreal', '25%')] = _hist_boreal_25
all_test_results[('boreal', '25%')] = _res_boreal_25



Using 25% of dataset:
  Total images: 1417
  Used images: 354
  Train: 247
  Val: 53
  Test: 54

TRAINING WITH 25% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.4529, F1: 0.7728, mIoU: 0.6423
  Val   - Loss: 0.4204, F1: 0.8237, mIoU: 0.7053
Epoch [5/50]
  Train - Loss: 0.3642, F1: 0.8800, mIoU: 0.7890
  Val   - Loss: 0.3597, F1: 0.8503, mIoU: 0.7427
Epoch [10/50]
  Train - Loss: 0.3173, F1: 0.9075, mIoU: 0.8320
  Val   - Loss: 0.3393, F1: 0.8951, mIoU: 0.8135
  ✓ Best model saved! (mIoU: 0.8135)
Epoch [15/50]
  Train - Loss: 0.2833, F1: 0.9120, mIoU: 0.8412
  Val   - Loss: 0.2885, F1: 0.8992, mIoU: 0.8201
  ✓ Best model saved! (mIoU: 0.8201)
Epoch [20/50]
  Train - Loss: 0.2490, F1: 0.9278, mIoU: 0.8666
  Val   - Loss: 0.2683, F1: 0.8913, mIoU: 0.8083
Epoch [25/50]
  Train - Loss: 0.2275, F1: 0.9294, mIoU: 0.8707
  Val   - Loss: 0.2377, F1: 0.9065, mIoU: 0.8316
Epoch [30/50]
  Train - Loss: 0.1885, F1: 0.9491, mIoU: 0.9044
  Val   - Loss: 0.2033, F1: 0.9097, mIoU: 0.8378
Epoch [

#### BOREAL — 50% training data

In [ ]:
_train_ds_boreal_50, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=0.5
)
_train_loader_boreal_50 = DataLoader(
    _train_ds_boreal_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_50, _res_boreal_50 = train_model(
    _train_loader_boreal_50, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=0.5,
)

all_histories[('boreal', '50%')] = _hist_boreal_50
all_test_results[('boreal', '50%')] = _res_boreal_50



Using 50% of dataset:
  Total images: 1417
  Used images: 708
  Train: 495
  Val: 106
  Test: 107

TRAINING WITH 50% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.4452, F1: 0.7885, mIoU: 0.6667
  Val   - Loss: 0.3823, F1: 0.8527, mIoU: 0.7461
Epoch [5/50]
  Train - Loss: 0.3341, F1: 0.8960, mIoU: 0.8144
  Val   - Loss: 0.3225, F1: 0.9012, mIoU: 0.8233
  ✓ Best model saved! (mIoU: 0.8233)
Epoch [10/50]
  Train - Loss: 0.2628, F1: 0.9130, mIoU: 0.8433
  Val   - Loss: 0.2539, F1: 0.9154, mIoU: 0.8466
  ✓ Best model saved! (mIoU: 0.8466)
Epoch [15/50]
  Train - Loss: 0.2009, F1: 0.9319, mIoU: 0.8743
  Val   - Loss: 0.1925, F1: 0.9246, mIoU: 0.8620
Epoch [20/50]
  Train - Loss: 0.1578, F1: 0.9416, mIoU: 0.8910
  Val   - Loss: 0.1664, F1: 0.9186, mIoU: 0.8530
Epoch [25/50]
  Train - Loss: 0.1288, F1: 0.9458, mIoU: 0.8981
  Val   - Loss: 0.1352, F1: 0.9252, mIoU: 0.8634
Epoch [30/50]
  Train - Loss: 0.0982, F1: 0.9549, mIoU: 0.9143
  Val   - Loss: 0.1173, F1: 0.9268, mIoU: 0.8662
Epoch

#### BOREAL — 100% training data

In [ ]:
_train_ds_boreal_100, _, _ = create_data_splits(
    DATASETS['boreal']['images'], DATASETS['boreal']['masks'],
    _tr_tf_boreal, _vl_tf_boreal, data_percentage=1.0
)
_train_loader_boreal_100 = DataLoader(
    _train_ds_boreal_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_boreal_100, _res_boreal_100 = train_model(
    _train_loader_boreal_100, _val_loader_boreal, _test_loader_boreal,
    dataset_name='boreal', data_percentage=1.0,
)

all_histories[('boreal', '100%')] = _hist_boreal_100
all_test_results[('boreal', '100%')] = _res_boreal_100



Using 100% of dataset:
  Total images: 1417
  Used images: 1417
  Train: 991
  Val: 212
  Test: 214

TRAINING WITH 100% OF DATASET — BOREAL

Epoch [1/50]
  Train - Loss: 0.4098, F1: 0.8287, mIoU: 0.7174
  Val   - Loss: 0.3678, F1: 0.8820, mIoU: 0.7918
Epoch [5/50]
  Train - Loss: 0.2729, F1: 0.9044, mIoU: 0.8285
  Val   - Loss: 0.2536, F1: 0.9125, mIoU: 0.8414
  ✓ Best model saved! (mIoU: 0.8414)
Epoch [10/50]
  Train - Loss: 0.1760, F1: 0.9277, mIoU: 0.8674
  Val   - Loss: 0.1699, F1: 0.9091, mIoU: 0.8353
Epoch [15/50]
  Train - Loss: 0.1187, F1: 0.9376, mIoU: 0.8848
  Val   - Loss: 0.1227, F1: 0.9246, mIoU: 0.8622
Epoch [20/50]
  Train - Loss: 0.0849, F1: 0.9461, mIoU: 0.8992
  Val   - Loss: 0.1010, F1: 0.9257, mIoU: 0.8650
Epoch [25/50]
  Train - Loss: 0.0696, F1: 0.9502, mIoU: 0.9063
  Val   - Loss: 0.0818, F1: 0.9333, mIoU: 0.8782
Epoch [30/50]
  Train - Loss: 0.0572, F1: 0.9548, mIoU: 0.9148
  Val   - Loss: 0.0749, F1: 0.9349, mIoU: 0.8800
  ✓ Best model saved! (mIoU: 0.8800)
Ep

### Dataset: KAGGLE

In [ ]:
_img_size_kaggle = DATASET_IMG_SIZE['kaggle']
_tr_tf_kaggle, _vl_tf_kaggle = get_transforms(_img_size_kaggle)
print(f"\nKAGGLE — resolution: {_img_size_kaggle}x{_img_size_kaggle}")

_, _val_ds_kaggle, _test_ds_kaggle = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=1.0
)
_val_loader_kaggle = DataLoader(_val_ds_kaggle,  batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
_test_loader_kaggle = DataLoader(_test_ds_kaggle, batch_size=BATCH_SIZE, shuffle=False,
                                    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)



KAGGLE — resolution: 256x256

Using 100% of dataset:
  Total images: 27460
  Used images: 27460
  Train: 19222
  Val: 4119
  Test: 4119


#### KAGGLE — 25% training data

In [ ]:
_train_ds_kaggle_25, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=0.25
)
_train_loader_kaggle_25 = DataLoader(
    _train_ds_kaggle_25, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_25, _res_kaggle_25 = train_model(
    _train_loader_kaggle_25, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=0.25,
)

all_histories[('kaggle', '25%')] = _hist_kaggle_25
all_test_results[('kaggle', '25%')] = _res_kaggle_25



Using 25% of dataset:
  Total images: 27460
  Used images: 6865
  Train: 4805
  Val: 1029
  Test: 1031

TRAINING WITH 25% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.8580, F1: 0.3767, mIoU: 0.2364
  Val   - Loss: 0.8115, F1: 0.4880, mIoU: 0.3241
Epoch [5/50]
  Train - Loss: 0.3403, F1: 0.7203, mIoU: 0.5645
  Val   - Loss: 0.3528, F1: 0.6916, mIoU: 0.5307
Epoch [10/50]
  Train - Loss: 0.2276, F1: 0.7787, mIoU: 0.6388
  Val   - Loss: 0.2613, F1: 0.7448, mIoU: 0.5951
Epoch [15/50]
  Train - Loss: 0.2033, F1: 0.7980, mIoU: 0.6648
  Val   - Loss: 0.2410, F1: 0.7600, mIoU: 0.6145
  ✓ Best model saved! (mIoU: 0.6145)
Epoch [20/50]
  Train - Loss: 0.1932, F1: 0.8072, mIoU: 0.6777
  Val   - Loss: 0.2337, F1: 0.7666, mIoU: 0.6232
  ✓ Best model saved! (mIoU: 0.6232)
Epoch [25/50]
  Train - Loss: 0.1832, F1: 0.8170, mIoU: 0.6913
  Val   - Loss: 0.2378, F1: 0.7623, mIoU: 0.6177
Epoch [30/50]
  Train - Loss: 0.1779, F1: 0.8223, mIoU: 0.6989
  Val   - Loss: 0.2297, F1: 0.7704, mIoU: 0.6281


#### KAGGLE — 50% training data

In [ ]:
_train_ds_kaggle_50, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=0.5
)
_train_loader_kaggle_50 = DataLoader(
    _train_ds_kaggle_50, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_50, _res_kaggle_50 = train_model(
    _train_loader_kaggle_50, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=0.5,
)

all_histories[('kaggle', '50%')] = _hist_kaggle_50
all_test_results[('kaggle', '50%')] = _res_kaggle_50



Using 50% of dataset:
  Total images: 27460
  Used images: 13730
  Train: 9611
  Val: 2059
  Test: 2060

TRAINING WITH 50% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.7981, F1: 0.4532, mIoU: 0.2998
  Val   - Loss: 0.6405, F1: 0.6159, mIoU: 0.4467
Epoch [5/50]
  Train - Loss: 0.2468, F1: 0.7594, mIoU: 0.6135
  Val   - Loss: 0.2580, F1: 0.7457, mIoU: 0.5965
Epoch [10/50]
  Train - Loss: 0.2109, F1: 0.7896, mIoU: 0.6534
  Val   - Loss: 0.2288, F1: 0.7715, mIoU: 0.6295
  ✓ Best model saved! (mIoU: 0.6295)
Epoch [15/50]
  Train - Loss: 0.1975, F1: 0.8027, mIoU: 0.6713
  Val   - Loss: 0.2204, F1: 0.7798, mIoU: 0.6405
Epoch [20/50]
  Train - Loss: 0.1878, F1: 0.8124, mIoU: 0.6848
  Val   - Loss: 0.2143, F1: 0.7858, mIoU: 0.6484
  ✓ Best model saved! (mIoU: 0.6484)
Epoch [25/50]
  Train - Loss: 0.1820, F1: 0.8181, mIoU: 0.6928
  Val   - Loss: 0.2176, F1: 0.7825, mIoU: 0.6439
Epoch [30/50]
  Train - Loss: 0.1782, F1: 0.8219, mIoU: 0.6983
  Val   - Loss: 0.2162, F1: 0.7839, mIoU: 0.6458

#### KAGGLE — 100% training data

In [ ]:
_train_ds_kaggle_100, _, _ = create_data_splits(
    DATASETS['kaggle']['images'], DATASETS['kaggle']['masks'],
    _tr_tf_kaggle, _vl_tf_kaggle, data_percentage=1.0
)
_train_loader_kaggle_100 = DataLoader(
    _train_ds_kaggle_100, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True
)

_hist_kaggle_100, _res_kaggle_100 = train_model(
    _train_loader_kaggle_100, _val_loader_kaggle, _test_loader_kaggle,
    dataset_name='kaggle', data_percentage=1.0,
)

all_histories[('kaggle', '100%')] = _hist_kaggle_100
all_test_results[('kaggle', '100%')] = _res_kaggle_100



Using 100% of dataset:
  Total images: 27460
  Used images: 27460
  Train: 19222
  Val: 4119
  Test: 4119

TRAINING WITH 100% OF DATASET — KAGGLE

Epoch [1/50]
  Train - Loss: 0.6534, F1: 0.5524, mIoU: 0.3924
  Val   - Loss: 0.3881, F1: 0.6906, mIoU: 0.5294
Epoch [5/50]
  Train - Loss: 0.2227, F1: 0.7780, mIoU: 0.6379
  Val   - Loss: 0.2243, F1: 0.7761, mIoU: 0.6355
  ✓ Best model saved! (mIoU: 0.6355)
Epoch [10/50]
  Train - Loss: 0.2024, F1: 0.7978, mIoU: 0.6647
  Val   - Loss: 0.2143, F1: 0.7858, mIoU: 0.6485
Epoch [15/50]
  Train - Loss: 0.1900, F1: 0.8101, mIoU: 0.6817
  Val   - Loss: 0.2044, F1: 0.7957, mIoU: 0.6619
  ✓ Best model saved! (mIoU: 0.6619)
Epoch [20/50]
  Train - Loss: 0.1824, F1: 0.8177, mIoU: 0.6923
  Val   - Loss: 0.2028, F1: 0.7973, mIoU: 0.6640
  ✓ Best model saved! (mIoU: 0.6640)
Epoch [25/50]
  Train - Loss: 0.1766, F1: 0.8234, mIoU: 0.7005
  Val   - Loss: 0.2042, F1: 0.7958, mIoU: 0.6622
Epoch [30/50]
  Train - Loss: 0.1730, F1: 0.8271, mIoU: 0.7058
  Val   